# Étape 17b — Deuxième jeu de données réel : DBE-KT22

**Objectif.** Vérifier, dans un contexte éducatif très différent de Junyi, que **la couverture du graphe pèse plus que la justesse de ses relations**, et que cela vaut pour PrereqKT **et** GKT.

| | Junyi (étapes 5 à 15) | DBE-KT22 (ici) |
|---|---|---|
| Matière, niveau | mathématiques, école | bases de données, université (ANU) |
| Données | 20 000 élèves, ~800 exercices | 1 264 étudiants, 212 questions, 162 000 réponses |
| Graphe expert | prérequis **orientés** entre exercices | relations **non orientées** entre notions (« notions liées ») |
| Ordre de travail | suit la carte des experts | libre |

**Le graphe expert entre questions.** La question A est reliée à la question B si une notion de A est liée (selon les experts) à une notion de B, et si A et B ne partagent aucune notion. Les relations étant symétriques, le graphe l'est aussi.
**Questions à plusieurs notions** (remarque de l'encadrant) : une question qui a plus de notions reçoit plus de voisins. Pour ne pas favoriser ces questions, **chaque question garde exactement le même nombre de voisins dans toutes les conditions** (comme à l'étape 13) : seules la couverture et la nature des voisins changent.

**Plan** (identique à l'étape 13) :
- **couverture** : 25 %, 50 %, 100 % des questions qui ont des voisins experts ;
- **source** des voisins : `expert` (vraies relations) · `theme_faux` (questions du **même quiz** sans relation experte ni notion commune) · `hors_theme_faux` (questions d'un **autre quiz**, sans relation) ;
- `aucun` (pas de graphe) comme référence ; le « thème » est le numéro de quiz (titre « Q10-20 » → quiz Q10).

**Modèles et régimes :** PrereqKT (10 conditions) et GKT (`aucun`, `expert` 25 % et 100 %, `theme_faux` et `hors_theme_faux` 100 %), avec **5 %** et **20 %** des étudiants d'entraînement (1 % ne ferait que 8 étudiants), **10 graines**.
Contrôle supplémentaire : PrereqKT `aucun` et `expert` avec **100 %** des étudiants (pour comparer à l'étude de Truong, 2026, qui mesure sur toutes les réponses avec toutes les données).

**Hypothèses** (mêmes définitions qu'aux étapes 13 et 14 ; tests t appariés bilatéraux par graine ; Holm sur l'ensemble des tests) :
H1 couverture (expert 100 % vs 25 %), H2 justesse (expert vs theme_faux à 100 %), H3 proximité (theme_faux vs hors_theme_faux à 100 %), H4 un graphe faux change-t-il la prédiction (hors_theme_faux 100 % vs aucun),
et comparaison directe de la **taille** des effets de couverture et de justesse.

### Deux comptes en parallèle
`PARTIE = 'A'` (graines 0 à 4) et `PARTIE = 'B'` (graines 5 à 9) écrivent chacune leur propre fichier. **GPU T4.**
La cellule D1 prépare les données (une seule fois, quelques secondes) : il faut que le dossier `projet_kt/dbe_kt22` contienne les fichiers de DBE-KT22 (décompressés par le notebook 17a).

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule D1 — Préparation de DBE-KT22 au même format que Junyi (dossier dbe_prepare)
import glob, re
DBE  = os.path.join(DOSSIER, 'dbe_kt22')
PREP = os.path.join(DOSSIER, 'dbe_prepare')          # toutes les cellules suivantes lisent ce dossier
os.makedirs(PREP, exist_ok=True)

def lire(nom):
    c = [p for p in glob.glob(os.path.join(DBE, '**', '*.csv'), recursive=True)
         if os.path.splitext(os.path.basename(p))[0].lower() == nom.lower()]
    assert c, f"{nom}.csv introuvable dans {DBE} : lancez d'abord la cellule 1 du notebook 17a (elle décompresse le zip)"
    return pd.read_csv(c[0], low_memory=False)

TR, QK, KR, QS = lire('Transaction'), lire('Question_KC_Relationships'), lire('KC_Relationships'), lire('Questions')
MIN_INTER, LONG_MAX, PARTS_ = 10, 200, {'train': 0.7, 'valid': 0.1, 'test': 0.2}

# Questions : indices 0..N-1, quiz (thème) tiré du titre « Q10-20 » -> « Q10 »
ids_q = sorted(set(QS['id']) | set(TR['question_id'].dropna().astype(int)))
idx_q = {q: i for i, q in enumerate(ids_q)}
titre = QS.set_index('id')['question_title'].to_dict()
def quiz(t):
    m = re.match(r'\s*(Q\d+)', str(t))
    return m.group(1) if m else np.nan
kc_par_q = QK.groupby('question_id')['knowledgecomponent_id'].apply(lambda s: set(map(int, s)))
q_par_kc = QK.groupby('knowledgecomponent_id')['question_id'].apply(lambda s: set(map(int, s)))
exos_d = pd.DataFrame({'exo_id': range(len(ids_q)), 'id_dbe': ids_q,
                       'name': [str(titre.get(q, q)) for q in ids_q],
                       'topic': [quiz(titre.get(q)) for q in ids_q],
                       'kcs': ['|'.join(map(str, sorted(kc_par_q.get(q, set())))) for q in ids_q]})

# Graphe expert entre questions (symétrique)
A_d = np.zeros((len(ids_q), len(ids_q)), dtype=np.float32)
for a_kc, b_kc in zip(KR['from_knowledgecomponent_id'], KR['to_knowledgecomponent_id']):
    for qa in q_par_kc.get(a_kc, ()):
        for qb in q_par_kc.get(b_kc, ()):
            if qa != qb and qa in idx_q and qb in idx_q and not (kc_par_q.get(qa, set()) & kc_par_q.get(qb, set())):
                A_d[idx_q[qa], idx_q[qb]] = 1

# Séquences : réponses dans l'ordre du temps, découpées en morceaux de 200
T_ = TR.dropna(subset=['student_id', 'question_id']).copy()
T_['t'] = pd.to_datetime(T_['start_time'], errors='coerce', utc=True)
T_ = T_.sort_values(['student_id', 't', 'id'], kind='stable')
T_['q'] = T_['question_id'].astype(int).map(idx_q)
T_['r'] = T_['answer_state'].astype(str).str.lower().isin(['true', '1', '1.0', 't'])
n_par = T_.groupby('student_id').size()
eleves = np.array(sorted(n_par[n_par >= MIN_INTER].index))
rng_ = np.random.default_rng(42); rng_.shuffle(eleves)
c1, c2 = int(PARTS_['train'] * len(eleves)), int((PARTS_['train'] + PARTS_['valid']) * len(eleves))
partie_de = {u: 'train' for u in eleves[:c1]}; partie_de.update({u: 'valid' for u in eleves[c1:c2]}); partie_de.update({u: 'test' for u in eleves[c2:]})
lignes = []
for u, g in T_[T_['student_id'].isin(partie_de)].groupby('student_id', sort=False):
    q, r = g['q'].values, g['r'].astype(int).values
    for d in range(0, len(q), LONG_MAX):
        if len(q[d:d + LONG_MAX]) >= 2:
            lignes.append({'uid': f'u{u}', 'partie': partie_de[u], 'morceau': d // LONG_MAX,
                           'exercices': ','.join(map(str, q[d:d + LONG_MAX])), 'reponses': ','.join(map(str, r[d:d + LONG_MAX]))})
seqs = pd.DataFrame(lignes)
for p_ in PARTS_:
    seqs[seqs.partie == p_].drop(columns='partie').to_csv(os.path.join(PREP, f'sequences_{p_}.csv'), index=False)
exos_d.to_csv(os.path.join(PREP, 'exercices.csv'), index=False)
np.save(os.path.join(PREP, 'adjacence_expert.npy'), A_d)

info = {'questions': len(ids_q), 'etudiants_gardes': len(eleves), 'reponses': int(T_['student_id'].isin(partie_de).sum()),
        'fleches_expertes_entre_questions': int(A_d.sum()), 'symetrique': bool((A_d == A_d.T).all()),
        'questions_avec_voisins_%': round(100 * float((A_d.sum(0) > 0).mean()), 1),
        'quiz_distincts': int(exos_d.topic.nunique()), 'questions_sans_quiz': int(exos_d.topic.isna().sum()),
        'fleches_expertes_dans_le_meme_quiz_%': round(100 * float(np.mean([exos_d.topic[a] == exos_d.topic[b] for a, b in zip(*np.nonzero(A_d))])), 1)}
json.dump(info, open(os.path.join(PREP, 'infos_preparation.json'), 'w'), indent=1)
print(json.dumps(info, indent=1, ensure_ascii=False))
print(seqs.groupby('partie').agg(etudiants=('uid', 'nunique'), sequences=('uid', 'size')))
print('Questions par quiz :', exos_d.topic.value_counts().to_dict())

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=1024):   # grands lots : même résultat, évaluation beaucoup plus rapide pour GKT
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = (GKT(N, hp, A) if nom_modele == 'GKT' else GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT'))).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule G1 — GKT (Nakagawa et al., 2019), variante à graphe fixe
HP_GKT = dict(HP, gkt_hidden=32, gkt_emb=32)

class GKT(nn.Module):
    def __init__(self, n, hp, A):
        super().__init__()
        dh, de = hp['gkt_hidden'], hp['gkt_emb']
        self.n, self.dh = n, dh
        A = np.asarray(A) > 0
        voisins = []
        for i in range(n):
            suc = [int(j) for j in np.nonzero(A[i])[0] if j != i]                  # i est prérequis de j
            vus = set(suc)
            pre = [int(j) for j in np.nonzero(A[:, i])[0] if j != i and j not in vus]   # j est prérequis de i
            voisins.append([(j, 1) for j in suc] + [(j, 2) for j in pre])
        K = max(1, max(map(len, voisins)))
        idx = np.full((n + 1, K), n, dtype=np.int64)        # n = nœud fictif servant de remplissage
        typ = np.zeros((n + 1, K), dtype=np.int64)          # 0 remplissage, 1 successeur, 2 prérequis
        for i, v in enumerate(voisins):
            for k, (j, t) in enumerate(v):
                idx[i, k], typ[i, k] = j, t
        self.register_buffer('NB', torch.from_numpy(idx))
        self.register_buffer('TY', torch.from_numpy(typ))
        self.h0 = nn.Parameter(torch.zeros(n + 1, dh))      # état initial appris de chaque exercice
        self.Ex = nn.Embedding(2 * n, de)                   # interaction (exercice, réponse)
        self.Ec = nn.Embedding(n + 1, de)                   # exercice
        mlp = lambda d_in: nn.Sequential(nn.Linear(d_in, dh), nn.ReLU(), nn.Dropout(hp['dropout']), nn.Linear(dh, dh))
        self.f_soi = mlp(dh + de)
        self.f_suc = mlp(2 * (dh + de))                     # message de i vers ses successeurs
        self.f_pre = mlp(2 * (dh + de))                     # message de i vers ses prérequis
        self.efface, self.ajoute = nn.Linear(dh, dh), nn.Linear(dh, dh)
        self.w_ea = nn.Parameter(torch.ones(n + 1))         # poids effacement-ajout propre à chaque exercice
        self.gru = nn.GRUCell(dh, dh)
        self.sortie = nn.Linear(dh, 1)

    def forward(self, q, r):
        B, L = q.shape
        h = self.h0.unsqueeze(0).expand(B, -1, -1).contiguous()
        lig = torch.arange(B, device=q.device)
        logits = []
        for t in range(L - 1):
            i = q[:, t]
            h_i = h[lig, i]
            soi = torch.cat([h_i, self.Ex(2 * i + r[:, t])], -1)
            nb, ty = self.NB[i], self.TY[i]                                     # (B, K)
            h_nb = torch.gather(h, 1, nb.unsqueeze(-1).expand(-1, -1, self.dh))
            paire = torch.cat([soi.unsqueeze(1).expand(-1, nb.size(1), -1), h_nb, self.Ec(nb)], -1)
            m_nb = torch.where((ty == 1).unsqueeze(-1), self.f_suc(paire), self.f_pre(paire))
            m = torch.cat([self.f_soi(soi).unsqueeze(1), m_nb], 1)              # (B, K+1, dh)
            cible = torch.cat([i.unsqueeze(1), nb], 1)
            w = self.w_ea[cible].unsqueeze(-1)
            m = m - w * torch.sigmoid(self.efface(m)) * m + w * torch.tanh(self.ajoute(m))
            h_old = torch.cat([h_i.unsqueeze(1), h_nb], 1)
            h_new = self.gru(m.reshape(-1, self.dh), h_old.reshape(-1, self.dh)).view_as(h_old)
            valide = torch.cat([torch.ones_like(ty[:, :1], dtype=torch.bool), ty > 0], 1).unsqueeze(-1)
            h_new = torch.where(valide, h_new, h_old)
            h = h.scatter(1, cible.unsqueeze(-1).expand(-1, -1, self.dh), h_new)
            logits.append(self.sortie(h[lig, q[:, t + 1]]).squeeze(-1))
        return torch.stack(logits, 1)

print('GKT prêt. Plus grand nombre de voisins (graphe expert) :', GKT(N, HP_GKT, A_expert).NB.shape[1])

In [ ]:
# Cellule D2 — Construction des 9 graphes factoriels (DBE-KT22) (déterministe pour chaque graine)
COUVERTURES = [0.25, 0.5, 1.0]
SOURCES = ['expert', 'theme_faux', 'hors_theme_faux']
GRAINES = list(range(10))

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
# DBE-KT22 : relations non orientées. « Liées » à B = voisins experts de B et questions qui partagent une notion avec B
kc_q = exos['kcs'].fillna('').astype(str).apply(lambda s: set(s.split('|')) - {''}).tolist()
ancetres = {b: set(G_exp.predecessors(b)) | set(G_exp.successors(b)) | {a for a in range(N) if a != b and kc_q[a] & kc_q[b]}
            for b in range(N)}
prereq_directs = {b: list(G_exp.predecessors(b)) for b in range(N)}
UNIVERS = sorted(b for b in range(N) if prereq_directs[b])
topic = exos.set_index('exo_id')['topic'].to_dict()
meme_theme = {}
for b in range(N):
    meme_theme[b] = [a for a in range(N) if a != b and isinstance(topic[b], str) and topic[a] == topic[b]]

def graphe_factoriel(couv, source, graine):
    rng = random.Random(90_000 + graine)
    ordre = UNIVERS[:]; rng.shuffle(ordre)                 # sous-ensembles emboîtés : 25 % ⊂ 50 % ⊂ 100 %
    choisis = ordre[:int(round(couv * len(ordre)))]
    E, manquants = [], 0
    for b in sorted(choisis):
        n_b = len(prereq_directs[b])
        if source == 'expert':
            E += [(a, b) for a in prereq_directs[b]]
            continue
        interdits = ancetres[b] | {b}
        if source == 'theme_faux':
            pool = [a for a in meme_theme[b] if a not in interdits]
        else:
            pool = [a for a in range(N) if a not in interdits and (not isinstance(topic[b], str) or topic[a] != topic[b])]
        rng_b = random.Random(f'{graine}-{source}-{b}')        # même tirage pour B à toutes les couvertures
        pris = rng_b.sample(pool, min(n_b, len(pool)))
        manquants += n_b - len(pris)
        E += [(a, b) for a in pris]
    return sorted(set(E)), manquants

# Vérifications : nombre de flèches, part des flèches entre exercices du même thème, couverture des premières tentatives
_, Qt, _, Ft = donnees['test']
premieres = pd.Series(np.concatenate([q[f] for q, f in zip(Qt, Ft)])).value_counts()
poids = premieres.reindex(range(N), fill_value=0).values
def couverture_1res(E):
    a = np.zeros(N, bool)
    for _, b in E:
        a[b] = True
    return 100 * (poids * a).sum() / poids.sum()

lignes = []
for couv in COUVERTURES:
    for source in SOURCES:
        m = []
        for g in GRAINES:
            E, manq = graphe_factoriel(couv, source, g)
            meme = np.mean([isinstance(topic[b], str) and topic[a] == topic[b] for a, b in E]) if E else 0
            juste = np.mean([a in ancetres[b] for a, b in E]) if E else 0
            m.append({'fleches': len(E), 'manquantes': manq, 'meme_theme_%': 100 * meme,
                      'prerequis_vrais_%': 100 * juste, 'couverture_1res_%': couverture_1res(E)})
        lignes.append({'couverture': couv, 'source': source, **pd.DataFrame(m).mean().round(1).to_dict()})
verif = pd.DataFrame(lignes)
verif.to_csv(os.path.join(RESULTATS, 'dbe_factoriel_graphes.csv'), index=False)
print(f'Univers : {len(UNIVERS)} questions avec au moins un voisin expert')
print(verif.to_string(index=False))

In [ ]:
# Cellule D3 — Les entraînements de votre partie, graine par graine (reprise automatique)
PARTIE = 'A'          # 'A' = graines 0 à 4 ; 'B' = graines 5 à 9 (sur l'autre compte, en même temps)

GRAINES_PARTIE = {'A': [0, 1, 2, 3, 4], 'B': [5, 6, 7, 8, 9]}[PARTIE]
PARTIES = ['A', 'B']
FRACTIONS = [0.2, 0.05]
COND_PKT = [('aucun', None, None)] + [(f'{s}_c{int(c * 100)}', c, s) for c in COUVERTURES for s in SOURCES]
COND_GKT = [('aucun', None, None), ('expert_c25', 0.25, 'expert'), ('expert_c100', 1.0, 'expert'),
            ('theme_faux_c100', 1.0, 'theme_faux'), ('hors_theme_faux_c100', 1.0, 'hors_theme_faux')]
PLAN = [(m, lib, c, s, f) for f in FRACTIONS for m, conds in [('PrereqKT', COND_PKT), ('GKT', COND_GKT)] for lib, c, s in conds]
PLAN += [('PrereqKT', 'aucun', None, None, 1.0), ('PrereqKT', 'expert_c100', 1.0, 'expert', 1.0)]   # contrôle « toutes les données »

chemin_d = os.path.join(RESULTATS, f'dbe_resultats_{PARTIE}.csv')
res_d = pd.read_csv(chemin_d).to_dict('records') if os.path.exists(chemin_d) else []
fait = set()
for p_ in PARTIES:
    c_ = os.path.join(RESULTATS, f'dbe_resultats_{p_}.csv')
    if os.path.exists(c_):
        fait |= {(x['modele'], x['condition'], float(x['fraction']), int(x['graine'])) for x in pd.read_csv(c_).to_dict('records')}
t0_ = time.time()
for g_ in GRAINES_PARTIE:
    for modele, lib, couv, source, frac in PLAN:
        if (modele, lib, frac, g_) in fait:
            continue
        A = graphe('aucun') if couv is None else vers_matrice(graphe_factoriel(couv, source, g_)[0])
        res = entrainer(modele, A, fraction=frac, graine=g_, hp=HP_GKT if modele == 'GKT' else HP)
        res_d.append({'modele': modele, 'condition': lib, 'couverture': couv or 0.0, 'source': source or 'aucun',
                      'fraction': frac, 'graine': g_, **res})
        pd.DataFrame(res_d).to_csv(chemin_d, index=False)
        print(f'  graine {g_} | {frac:.0%} | {modele:8s} | {lib:22s} | AUC 1res = {res["AUC_1re_test"]} | {res["epoques"]} époques | {res["minutes"]} min')
    print(f'Graine {g_} terminée ({(time.time() - t0_) / 60:.0f} min écoulées)')
print(f'Partie {PARTIE} terminée.')

In [ ]:
# Cellule D4 — Analyse (quand les deux parties sont finies ; marche aussi avec les graines déjà terminées)
#   Après une déconnexion : relancez les cellules 1 à D2, puis celle-ci.
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
col = 'AUC_1re_test'
tab = pd.concat([pd.read_csv(os.path.join(RESULTATS, f'dbe_resultats_{p}.csv')) for p in ['A', 'B']
                 if os.path.exists(os.path.join(RESULTATS, f'dbe_resultats_{p}.csv'))], ignore_index=True)
tab = tab.drop_duplicates(['modele', 'condition', 'fraction', 'graine'])
attendu = 2 * (10 + 5) + 2
n_ = tab.groupby('graine').size()
graines_ok = sorted(n_[n_ == attendu].index)
tab = tab[tab.graine.isin(graines_ok)]
print(f'Graines complètes analysées : {graines_ok}')

def ic_bootstrap(d, n=10_000, graine=0):
    rng = np.random.default_rng(graine); d = np.asarray(d, dtype=float)
    return np.percentile(rng.choice(d, size=(n, len(d)), replace=True).mean(1), [2.5, 97.5])

# Référence « difficulté seule » (taux de réussite lissé de chaque question chez les étudiants d'entraînement)
_, Qt, Rt, Ft = donnees['test']
q_te = np.concatenate([q[1:] for q in Qt]); r_te = np.concatenate([r[1:] for r in Rt]); f_te = np.concatenate([f[1:] for f in Ft])
diff = []
for frac in [0.2, 0.05, 1.0]:
    for g_ in graines_ok:
        q_tr, r_tr, _ = sous_ensemble_train(frac, g_)
        qa, ra = np.concatenate(q_tr), np.concatenate(r_tr)
        n__ = np.bincount(qa, minlength=N); s__ = np.bincount(qa, weights=ra, minlength=N)
        taux = (s__ + 5 * s__.sum() / n__.sum()) / (n__ + 5)
        diff.append({'fraction': frac, 'graine': g_, 'AUC_1re': roc_auc_score(r_te[f_te], taux[q_te][f_te]), 'AUC_tout': roc_auc_score(r_te, taux[q_te])})
diff = pd.DataFrame(diff)

synthese, hyp, tailles, anova_l = [], [], [], []
for modele in ['PrereqKT', 'GKT']:
    for frac in [0.2, 0.05]:
        piv = tab[(tab.modele == modele) & (tab.fraction == frac)].pivot_table(index='graine', columns='condition', values=col)
        base = piv['aucun']; gain_exp = piv['expert_c100'] - base
        for lib in [c for c in piv.columns]:
            d = piv[lib] - base
            lo, hi = ic_bootstrap(d) if lib != 'aucun' else (0, 0)
            synthese.append({'modele': modele, 'fraction': frac, 'condition': lib, 'AUC_1re': round(piv[lib].mean(), 4),
                             'gain_vs_aucun': round(d.mean(), 4), 'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4),
                             'part_du_gain_expert_%': round(100 * d.mean() / gain_exp.mean(), 1) if abs(gain_exp.mean()) > 1e-9 else None})
        for h_, a, b in [('H1 couverture', 'expert_c100', 'expert_c25'), ('H2 justesse', 'expert_c100', 'theme_faux_c100'),
                         ('H3 proximité', 'theme_faux_c100', 'hors_theme_faux_c100'), ('H4 graphe faux vs aucun', 'hors_theme_faux_c100', 'aucun')]:
            d = piv[a] - piv[b]
            hyp.append({'modele': modele, 'fraction': frac, 'hypothese': f'{h_} : {a} − {b}', 'ecart_moyen': round(d.mean(), 4),
                        'graines_positives': f'{int((d > 0).sum())}/{len(d)}', 'p': stats.ttest_1samp(d, 0).pvalue})
        sens = np.sign(gain_exp.mean()) or 1.0
        cov = sens * (piv['expert_c100'] - piv['expert_c25']); jus = sens * (piv['expert_c100'] - piv['theme_faux_c100'])
        lo, hi = ic_bootstrap(cov - jus)
        tailles.append({'modele': modele, 'fraction': frac, 'sens_effet_graphe': 'gain' if sens > 0 else 'perte',
                        'effet_couverture': round(cov.mean(), 4), 'effet_justesse': round(jus.mean(), 4),
                        'couverture_moins_justesse': round((cov - jus).mean(), 4), 'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4),
                        'graines_couverture_plus_forte': f'{int(((cov - jus) > 0).sum())}/{len(cov)}',
                        'p': round(stats.ttest_1samp(cov - jus, 0).pvalue, 4)})
        if modele == 'PrereqKT':
            long_ = tab[(tab.modele == modele) & (tab.fraction == frac) & (tab.condition != 'aucun')].copy()
            long_['gain'] = long_[col].values - long_['graine'].map(base).values
            a_ = anova_lm(smf.ols('gain ~ C(couverture) * C(source) + C(graine)', data=long_).fit(), typ=2)
            for terme in ['C(couverture)', 'C(source)', 'C(couverture):C(source)']:
                anova_l.append({'fraction': frac, 'facteur': terme.replace('C(', '').replace(')', ''), 'F': round(a_.loc[terme, 'F'], 2),
                                'p': round(a_.loc[terme, 'PR(>F)'], 4),
                                'eta2_partiel': round(a_.loc[terme, 'sum_sq'] / (a_.loc[terme, 'sum_sq'] + a_.loc['Residual', 'sum_sq']), 3)})
for frac in [0.2, 0.05]:
    dd = diff[diff.fraction == frac].set_index('graine')['AUC_1re']
    synthese.append({'modele': 'difficulté seule', 'fraction': frac, 'condition': '-', 'AUC_1re': round(dd.mean(), 4)})

synthese, hyp, tailles, anova_t = pd.DataFrame(synthese), pd.DataFrame(hyp), pd.DataFrame(tailles), pd.DataFrame(anova_l)
o = hyp['p'].sort_values(); m = len(o)
hyp['p_holm'] = (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(hyp.index)
hyp['significatif_holm_5%'] = hyp['p_holm'] < 0.05
hyp[['p', 'p_holm']] = hyp[['p', 'p_holm']].round(4)

# Contrôle « toutes les données » (100 % des étudiants) : PrereqKT expert − aucun, sur les 1res tentatives et sur toutes les réponses
p100 = tab[(tab.fraction == 1.0) & (tab.modele == 'PrereqKT')].pivot_table(index='graine', columns='condition', values=['AUC_1re_test', 'AUC_test'])
tout = pd.DataFrame([{'mesure': m_, 'aucun': round(p100[(m_, 'aucun')].mean(), 4), 'expert_c100': round(p100[(m_, 'expert_c100')].mean(), 4),
                      'ecart': round((p100[(m_, 'expert_c100')] - p100[(m_, 'aucun')]).mean(), 4),
                      'p': round(stats.ttest_1samp(p100[(m_, 'expert_c100')] - p100[(m_, 'aucun')], 0).pvalue, 4)}
                     for m_ in ['AUC_1re_test', 'AUC_test']])

for nom_, t_ in [('dbe_synthese', synthese), ('dbe_hypotheses', hyp), ('dbe_tailles_effets', tailles), ('dbe_anova', anova_t),
                 ('dbe_toutes_donnees', tout), ('dbe_difficulte', diff)]:
    t_.to_csv(os.path.join(RESULTATS, f'{nom_}.csv'), index=False)
print('===== Gain d\'AUC (1res tentatives) par rapport à « aucun » =====\n', synthese.to_string(index=False))
print('\n===== ANOVA PrereqKT (bloc = graine) =====\n', anova_t.to_string(index=False))
print('\n===== Hypothèses (bilatérales, Holm sur', m, 'tests) =====\n', hyp.to_string(index=False))
print('\n===== Taille des effets : couverture contre justesse =====\n', tailles.to_string(index=False))
print('\n===== Contrôle « toutes les données » (PrereqKT, 100 % des étudiants) =====\n', tout.to_string(index=False))

# Figures
NOMS = {'expert': 'vraies relations (expert)', 'theme_faux': 'fausses, même quiz', 'hors_theme_faux': 'fausses, autre quiz'}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
for ax, frac in zip(axes, [0.2, 0.05]):
    s = synthese[(synthese.modele == 'PrereqKT') & (synthese.fraction == frac)]
    for so, mk in zip(SOURCES, ['o', 's', '^']):
        t = s[s.condition.str.startswith(so + '_c')].copy()
        t['c'] = t.condition.str.extract(r'_c(\d+)$')[0].astype(int)
        t = t.sort_values('c')
        ax.plot(t.c, t.gain_vs_aucun, marker=mk, label=NOMS[so]); ax.fill_between(t.c, t.IC95_bas, t.IC95_haut, alpha=0.15)
    ax.axhline(0, ls=':', c='black', lw=1); ax.set_xticks([25, 50, 100])
    ax.set_xlabel('Couverture (% des questions qui reçoivent des voisins)'); ax.set_title(f'DBE-KT22, PrereqKT — {frac:.0%} des étudiants')
axes[0].set_ylabel("Gain d'AUC sur les 1res tentatives\n(par rapport à « aucun graphe »)"); axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(RESULTATS, 'dbe_factoriel.png'), dpi=200); plt.show()

NOMS2 = {'expert_c25': 'expert\n25 %', 'expert_c100': 'expert\n100 %', 'theme_faux_c100': 'faux même\nquiz 100 %', 'hors_theme_faux_c100': 'faux autre\nquiz 100 %'}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), sharey=True)
for ax, frac in zip(axes, [0.2, 0.05]):
    x = np.arange(len(NOMS2))
    for nom, dec in [('PrereqKT', -0.2), ('GKT', 0.2)]:
        s = synthese[(synthese.modele == nom) & (synthese.fraction == frac)].set_index('condition').loc[list(NOMS2)]
        err = np.vstack([s.gain_vs_aucun - s.IC95_bas, s.IC95_haut - s.gain_vs_aucun]).astype(float).clip(min=0)
        ax.bar(x + dec, s['gain_vs_aucun'].astype(float), width=0.4, yerr=err, capsize=3, label=nom)
    ax.set_xticks(x); ax.set_xticklabels(NOMS2.values(), fontsize=8); ax.axhline(0, c='black', lw=0.8)
    ax.set_title(f'DBE-KT22 — {frac:.0%} des étudiants')
axes[0].set_ylabel("Gain d'AUC sur les 1res tentatives\n(par rapport à « aucun graphe »)"); axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(RESULTATS, 'dbe_gkt.png'), dpi=200); plt.show()

In [ ]:
# Cellule D5 — Rassembler tous les résultats de l'étape 17b dans un zip et le télécharger
import zipfile
FICHIERS = ['dbe_resultats_A.csv', 'dbe_resultats_B.csv', 'dbe_synthese.csv', 'dbe_hypotheses.csv', 'dbe_tailles_effets.csv',
            'dbe_anova.csv', 'dbe_toutes_donnees.csv', 'dbe_difficulte.csv', 'dbe_factoriel_graphes.csv',
            'dbe_factoriel.png', 'dbe_gkt.png']
chemin_zip = '/content/resultats_etape17b.zip'
with zipfile.ZipFile(chemin_zip, 'w') as z:
    for f in FICHIERS:
        c = os.path.join(RESULTATS, f)
        if os.path.exists(c):
            z.write(c, arcname=f); print('ajouté   :', f)
        else:
            print('MANQUANT :', f)
    c = os.path.join(PREP, 'infos_preparation.json')
    if os.path.exists(c):
        z.write(c, arcname='dbe_infos_preparation.json'); print('ajouté   : dbe_infos_preparation.json')
try:
    from google.colab import files
    files.download(chemin_zip)
except ImportError:
    print('Hors de Colab : le fichier est ici ->', chemin_zip)